In [2]:
!python -u lab07_starter.py

train (1438, 64) test (359, 64)
autoencoder test reconstruction MSE: 0.0163
feature network accuracy on real test images: 0.972
autoencoder samples from N(0, I): {'frechet': 86.35, 'precision': 0.435, 'recall': 0.772, 'nearest_train': 2.028}
saved starter_check.png


In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import lab07_starter as S

Xtr, ytr, Xte, yte = S.load_task()
print("Train:", Xtr.shape, "Test:", Xte.shape)

Train: torch.Size([1438, 64]) Test: torch.Size([359, 64])


# 1 - VAE

In [16]:
class VAE(nn.Module):
    def __init__(self, d_in=64, d_hidden=64, d_latent=8):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU())
        self.mu_layer = nn.Linear(d_hidden, d_latent)
        self.logvar_layer = nn.Linear(d_hidden, d_latent)
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, d_hidden), nn.ReLU(), nn.Linear(d_hidden, d_in)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.mu_layer(h), self.logvar_layer(h)

    def reparameterize(self, mu, logvar):
        eps = torch.randn_like(mu)
        return mu + torch.exp(0.5 * logvar) * eps

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

In [17]:
def kl_per_image(mu, logvar):
    """KL[q(z|x) || N(0,I)]: sum over the eight latent dimensions."""
    return -0.5 * (1 + logvar - mu.square() - logvar.exp()).sum(dim=1)


def reconstruction_per_image(logits, x):
    """Bernoulli negative log likelihood: sum over all 64 pixels."""
    return F.binary_cross_entropy_with_logits(logits, x, reduction="none").sum(dim=1)


def vae_loss(model, xb, yb=None, beta=1.0):
    """Mean over images of [sum_pixel BCE + beta * sum_latent KL]."""
    logits, mu, logvar = model(xb)
    return (reconstruction_per_image(logits, xb) + beta * kl_per_image(mu, logvar)).mean()

In [18]:
# Check 1: Closed-form KL vs Monte Carlo KL
torch.manual_seed(123)

mu = torch.randn(5, 8) * 0.5
logvar = torch.randn(5, 8) * 0.5

eps = torch.randn(50000, 5, 8)
z = mu.unsqueeze(0) + torch.exp(0.5 * logvar).unsqueeze(0) * eps

mc_kl = 0.5 * (
    z**2 - eps**2 - logvar.unsqueeze(0)
).sum(dim=2).mean(dim=0)

exact_kl = kl_per_image(mu, logvar)

print("KL difference:", (mc_kl - exact_kl).abs().max().item())
assert torch.allclose(mc_kl, exact_kl, atol=0.05, rtol=0)

KL difference: 0.02042853832244873


In [19]:
# Check 2: Gradient with and without reparameterization
torch.manual_seed(0)
model = VAE()
x = Xtr[:32]

logits, mu, logvar = model(x)
recon = reconstruction_per_image(logits, x).mean()

grad = torch.autograd.grad(recon, model.mu_layer.weight)[0]
with_trick = grad.norm().item()

z = model.reparameterize(mu, logvar).detach()
recon_detached = reconstruction_per_image(model.decode(z), x).mean()

grad_detached = torch.autograd.grad(
    recon_detached,
    model.mu_layer.weight,
    allow_unused=True
)[0]

without_trick = 0.0 if grad_detached is None else grad_detached.norm().item()

print("Gradient with trick:", with_trick)
print("Gradient without trick:", without_trick)

assert with_trick > 0
assert without_trick == 0.0

Gradient with trick: 0.8507521152496338
Gradient without trick: 0.0


In [20]:
# Check 3: Beta = 1 equals the negative ELBO
logits, mu, logvar = model(x)

recon = reconstruction_per_image(logits, x)
kl = kl_per_image(mu, logvar)

loss = (recon + kl).mean()

log_px = (
    x * F.logsigmoid(logits)
    + (1 - x) * F.logsigmoid(-logits)
).sum(dim=1)

negative_elbo = -(log_px - kl).mean()

print("VAE loss:", loss.item())
print("Negative ELBO:", negative_elbo.item())

assert torch.allclose(loss, negative_elbo, atol=1e-5)

VAE loss: 44.81682586669922
Negative ELBO: 44.81682586669922


# 2 - Autoencoder vs VAE

In [21]:
SEEDS = (0, 1, 2)
BETAS = (0.1, 0.5, 1.0, 2.0, 4.0)

EPOCHS = 200
LR = 2e-3
BATCH_SIZE = 64
D_LATENT = 8
D_HIDDEN = 64
N_SAMPLES = len(Xte)

net = S.train_feature_net(Xtr, ytr, seed=0)
net.eval()
fte = S.get_features(net, Xte)

def draw_prior(seed, n):
    g = torch.Generator().manual_seed(1000 + seed)
    return torch.randn(n, D_LATENT, generator=g)


@torch.no_grad()
def vae_samples(model, z):
    return model.decode(z).sigmoid()


@torch.no_grad()
def vae_test_mse(model):
    mu, _ = model.encode(Xte)
    return ((model.decode(mu).sigmoid() - Xte)**2).mean().item()

In [22]:
ae_models = {}
vae_models = {}
results = []
spreads = []

for seed in SEEDS:

    torch.manual_seed(seed)
    ae = S.AE(d_hidden=D_HIDDEN, d_latent=D_LATENT)
    S.train(
        ae, S.ae_loss, Xtr,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, seed=seed
    )
    ae.eval()
    ae_models[seed] = ae

    torch.manual_seed(seed)
    vae = VAE(d_hidden=D_HIDDEN, d_latent=D_LATENT)
    S.train(
        vae, vae_loss, Xtr,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, seed=seed
    )
    vae.eval()
    vae_models[seed] = vae

    z = draw_prior(seed, N_SAMPLES)

    with torch.no_grad():
        ae_generated = ae.decode(z)
        vae_generated = vae_samples(vae, z)

        ae_codes = ae.encode(Xtr)
        vae_codes = vae.encode(Xtr)[0]

    for name, samples, mse, codes in [
        ("AE", ae_generated, S.reconstruction_mse(ae, Xte), ae_codes),
        ("VAE", vae_generated, vae_test_mse(vae), vae_codes)
    ]:
        scores = S.score_samples(samples, fte, net, Xtr)

        results.append({
            "seed": seed,
            "model": name,
            "test_mse": mse,
            **scores
        })

        stds = codes.std(dim=0, unbiased=False).tolist()

        spreads.append({
            "seed": seed,
            "model": name,
            **{f"dim_{i}": std for i, std in enumerate(stds)}
        })

    S.save_grid(
        ae_generated[:30],
        f"part2_ae_seed{seed}.png",
        title=f"AE seed {seed}"
    )

    S.save_grid(
        vae_generated[:30],
        f"part2_vae_seed{seed}.png",
        title=f"VAE seed {seed}"
    )

    print("Finished seed", seed)

Finished seed 0
Finished seed 1
Finished seed 2


In [23]:
part2_df = pd.DataFrame(results)
spread_df = pd.DataFrame(spreads)

print("Reconstruction and sample quality:")
display(part2_df.round(4))

print("Latent code standard deviation:")
display(spread_df.round(4))

Reconstruction and sample quality:


,seed,model,test_mse,frechet,precision,recall,nearest_train
0,0,AE,0.0152,70.8812,0.3510,0.8663,2.0602
1,0,VAE,0.0259,12.9154,0.9136,0.4150,1.1303
2,1,AE,0.0151,87.3034,0.3593,0.7660,2.0956
3,1,VAE,0.0265,11.3869,0.8997,0.4680,1.1429
4,2,AE,0.0149,110.7445,0.4290,0.6574,2.3304
5,2,VAE,0.0263,13.4287,0.8830,0.4875,1.1366


Latent code standard deviation:


,seed,model,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7
0,0,AE,0.6831,0.8677,0.5646,0.6289,0.6887,0.8061,0.5306,0.6687
1,0,VAE,0.9400,0.9608,0.9366,0.4813,0.0216,0.8235,0.0206,0.0340
2,1,AE,0.6299,0.5758,0.7564,0.6445,1.0876,0.6603,0.7445,0.6449
3,1,VAE,0.9037,0.4447,0.8114,0.9113,0.8938,0.0214,0.0249,0.0377
4,2,AE,0.7149,0.7585,0.7503,0.7040,0.6167,0.7058,0.7707,0.7262
5,2,VAE,0.8947,0.9128,0.9039,0.0263,0.5364,0.8178,0.0302,0.0217


# 3 - KL weight

In [24]:
beta_models = {}
sweep_results = []

for beta in BETAS:
    for seed in SEEDS:

        if beta == 1.0:
            model = vae_models[seed]
        else:
            torch.manual_seed(seed)
            model = VAE(d_hidden=D_HIDDEN, d_latent=D_LATENT)

            S.train(
                model,
                lambda m, x, y, b=beta: vae_loss(m, x, y, b),
                Xtr,
                epochs=EPOCHS,
                lr=LR,
                batch_size=BATCH_SIZE,
                seed=seed
            )
            model.eval()

        beta_models[(beta, seed)] = model

        with torch.no_grad():
            mu, logvar = model.encode(Xtr)

            g = torch.Generator().manual_seed(7000 + seed)
            eps = torch.randn(mu.shape, generator=g)
            z = mu + torch.exp(0.5 * logvar) * eps

            recon = reconstruction_per_image(
                model.decode(z), Xtr
            ).mean().item()

            kl = kl_per_image(mu, logvar).mean().item()

            active = (mu.var(dim=0, unbiased=False) > 0.01).sum().item()

            samples = vae_samples(
                model, draw_prior(seed, N_SAMPLES)
            )

        scores = S.score_samples(samples, fte, net, Xtr)

        sweep_results.append({
            "beta": beta,
            "seed": seed,
            "reconstruction": recon,
            "kl": kl,
            "active_dimensions": active,
            "frechet": scores["frechet"],
            "precision": scores["precision"],
            "recall": scores["recall"]
        })

        S.save_grid(
            samples[:30],
            f"part3_beta{beta:g}_seed{seed}.png",
            title=f"Beta={beta}, seed={seed}"
        )

        print(f"Finished beta={beta}, seed={seed}")


sweep_df = pd.DataFrame(sweep_results)
display(sweep_df.round(4))

Finished beta=0.1, seed=0
Finished beta=0.1, seed=1
Finished beta=0.1, seed=2
Finished beta=0.5, seed=0
Finished beta=0.5, seed=1
Finished beta=0.5, seed=2
Finished beta=1.0, seed=0
Finished beta=1.0, seed=1
Finished beta=1.0, seed=2
Finished beta=2.0, seed=0
Finished beta=2.0, seed=1
Finished beta=2.0, seed=2
Finished beta=4.0, seed=0
Finished beta=4.0, seed=1
Finished beta=4.0, seed=2


,beta,seed,reconstruction,kl,active_dimensions,frechet,precision,recall
0,0.1,0,16.6792,14.2478,8,10.2701,0.7437,0.8468
1,0.1,1,16.5469,14.1241,8,13.9883,0.6964,0.8719
2,0.1,2,16.6033,14.2789,8,9.5870,0.6825,0.8217
3,0.5,0,18.7538,5.8431,7,9.5251,0.8189,0.7131
4,0.5,1,18.7185,5.9717,7,7.3731,0.8552,0.7326
5,0.5,2,18.5739,6.1659,7,8.8399,0.8440,0.7911
6,1.0,0,20.4055,3.5089,5,12.9154,0.9136,0.4150
7,1.0,1,20.7338,3.2489,5,11.3869,0.8997,0.4680
8,1.0,2,20.6249,3.3195,5,13.4287,0.8830,0.4875
9,2.0,0,23.8451,1.2772,3,43.7567,0.9415,0.0613


In [25]:
collapsed_config = {}

for seed in SEEDS:
    rows = sweep_df[sweep_df["seed"] == seed]

    row = rows.sort_values(
        ["active_dimensions", "beta"],
        ascending=[True, False]
    ).iloc[0]

    beta = row["beta"]
    active = int(row["active_dimensions"])

    collapsed_config[seed] = beta

    if active == 0:
        print(f"Seed {seed}: collapsed at beta={beta}")
    else:
        print(f"Seed {seed}: no full collapse (minimum {active} active dimensions)")

    samples = vae_samples(
        beta_models[(beta, seed)],
        draw_prior(seed, N_SAMPLES)
    )

    S.save_grid(
        samples[:30],
        f"part3_collapsed_seed{seed}.png",
        title=f"Beta={beta}, active={active}"
    )

Seed 0: collapsed at beta=4.0
Seed 1: collapsed at beta=4.0
Seed 2: collapsed at beta=4.0


# 4 - Probing the latent space

In [26]:
rng = np.random.default_rng(42)

pairs = []

while len(pairs) < 40:
    i, j = rng.choice(len(Xte), size=2, replace=False)

    if yte[i] != yte[j]:
        pairs.append((i, j))

xa = Xte[[i for i, j in pairs]]
xb = Xte[[j for i, j in pairs]]

t = torch.linspace(0, 1, 10).reshape(1, 10, 1)


@torch.no_grad()
def score_paths(images):
    probs = net(images).softmax(dim=1)

    confidence = probs.max(dim=1).values.mean().item()
    labels = probs.argmax(dim=1).reshape(40, 10)

    changes = (labels[:, 1:] != labels[:, :-1]).sum().item()

    return confidence, changes

In [27]:
interpolation_results = []

for seed in SEEDS:
    ae = ae_models[seed]
    vae = vae_models[seed]

    with torch.no_grad():
        za1, za2 = ae.encode(xa), ae.encode(xb)
        zv1 = vae.encode(xa)[0]
        zv2 = vae.encode(xb)[0]

        ae_path = (
            (1 - t) * za1[:, None, :]
            + t * za2[:, None, :]
        ).reshape(-1, D_LATENT)

        vae_path = (
            (1 - t) * zv1[:, None, :]
            + t * zv2[:, None, :]
        ).reshape(-1, D_LATENT)

        pixel_path = (
            (1 - t) * xa[:, None, :]
            + t * xb[:, None, :]
        ).reshape(-1, 64)

        ae_images = ae.decode(ae_path).clamp(0, 1)
        vae_images = vae.decode(vae_path).sigmoid()

    for name, images in [
        ("AE latent", ae_images),
        ("VAE latent", vae_images),
        ("Pixel cross-fade", pixel_path)
    ]:
        confidence, changes = score_paths(images)

        interpolation_results.append({
            "seed": seed,
            "method": name,
            "confidence": confidence,
            "class_changes": changes,
            "changes_per_pair": changes / 40
        })

        S.save_grid(
            images[:10],
            f"part4_{name.replace(' ', '_')}_seed{seed}.png",
            ncols=10,
            title=f"{name}, seed={seed}"
        )

In [28]:
interpolation_df = pd.DataFrame(interpolation_results)
display(interpolation_df.round(4))

for seed in SEEDS:
    rows = interpolation_df[interpolation_df.seed == seed]

    vae_conf = rows[rows.method == "VAE latent"]["confidence"].iloc[0]
    pixel_conf = rows[rows.method == "Pixel cross-fade"]["confidence"].iloc[0]

    print(
        f"Seed {seed}: VAE confidence={vae_conf:.4f}, "
        f"pixel confidence={pixel_conf:.4f}"
    )

,seed,method,confidence,class_changes,changes_per_pair
0,0,AE latent,0.9261,70,1.750
1,0,VAE latent,0.9287,63,1.575
2,0,Pixel cross-fade,0.9338,51,1.275
3,1,AE latent,0.9276,63,1.575
4,1,VAE latent,0.9287,61,1.525
5,1,Pixel cross-fade,0.9338,51,1.275
6,2,AE latent,0.9248,69,1.725
7,2,VAE latent,0.9298,63,1.575
8,2,Pixel cross-fade,0.9338,51,1.275


Seed 0: VAE confidence=0.9287, pixel confidence=0.9338
Seed 1: VAE confidence=0.9287, pixel confidence=0.9338
Seed 2: VAE confidence=0.9298, pixel confidence=0.9338


In [30]:
class CVAE(nn.Module):
    def __init__(self, d_hidden=64, d_latent=8):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(74, d_hidden),
            nn.ReLU()
        )

        self.mu_layer = nn.Linear(d_hidden, d_latent)
        self.logvar_layer = nn.Linear(d_hidden, d_latent)

        self.decoder = nn.Sequential(
            nn.Linear(d_latent + 10, d_hidden),
            nn.ReLU(),
            nn.Linear(d_hidden, 64)
        )

    def encode(self, x, y):
        h = self.encoder(torch.cat([x, S.one_hot(y)], dim=1))
        return self.mu_layer(h), self.logvar_layer(h)

    def decode(self, z, y):
        return self.decoder(torch.cat([z, S.one_hot(y)], dim=1))

    def forward(self, x, y):
        mu, logvar = self.encode(x, y)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return self.decode(z, y), mu, logvar


def cvae_loss(model, xb, yb):
    logits, mu, logvar = model(xb, yb)

    recon = reconstruction_per_image(logits, xb)
    kl = kl_per_image(mu, logvar)

    return (recon + kl).mean()

In [31]:
conditional_results = []
unconditional_results = []

for seed in SEEDS:
    torch.manual_seed(seed)
    model = CVAE()

    S.train(
        model, cvae_loss, Xtr, ytr,
        epochs=EPOCHS,
        lr=LR,
        batch_size=BATCH_SIZE,
        seed=seed
    )
    model.eval()

    grid_images = []

    with torch.no_grad():
        for digit in range(10):
            g = torch.Generator().manual_seed(
                50000 + 10 * seed + digit
            )
            z = torch.randn(200, D_LATENT, generator=g)
            labels = torch.full((200,), digit, dtype=torch.long)

            images = model.decode(z, labels).sigmoid()
            predictions = net(images).argmax(dim=1)

            share = (predictions == digit).float().mean().item()

            conditional_results.append({
                "seed": seed,
                "digit": digit,
                "requested_share": share
            })

            grid_images.append(images[:3])

        z = draw_prior(seed, 1000)
        images = vae_samples(vae_models[seed], z)

        predictions = net(images).argmax(dim=1)
        counts = torch.bincount(predictions, minlength=10)

        for digit, count in enumerate(counts.tolist()):
            unconditional_results.append({
                "seed": seed,
                "digit": digit,
                "count": count
            })

    S.save_grid(
        torch.cat(grid_images),
        f"part4_conditional_seed{seed}.png",
        title=f"Conditional VAE seed {seed}"
    )

In [32]:
conditional_df = pd.DataFrame(conditional_results)
unconditional_df = pd.DataFrame(unconditional_results)

print("Conditional VAE accuracy by requested digit:")
display(
    conditional_df.pivot(
        index="seed",
        columns="digit",
        values="requested_share"
    ).round(3)
)

print("Mean requested-digit share:")
display(conditional_df.groupby("seed")["requested_share"].mean())

print("Unconditional VAE class counts:")
display(
    unconditional_df.pivot(
        index="seed",
        columns="digit",
        values="count"
    )
)

Conditional VAE accuracy by requested digit:


digit,0,1,2,3,4,5,6,7,8,9
seed,,,,,,,,,,
0,0.995,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.995,1.0
1,1.000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.000,1.0
2,1.000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.000,1.0


Mean requested-digit share:


,requested_share
seed,
0,0.999
1,1.000
2,1.000


Unconditional VAE class counts:


digit,0,1,2,3,4,5,6,7,8,9
seed,,,,,,,,,,
0,90,83,78,135,94,87,126,109,119,79
1,97,78,80,119,97,111,114,76,133,95
2,95,96,88,140,95,94,88,89,111,104


# 5 - Metric Table

In [33]:
metric_results = []

for seed in SEEDS:
    rng = np.random.default_rng(9000 + seed)
    idx = rng.choice(len(Xtr), size=N_SAMPLES, replace=False)

    real = Xtr[idx]

    g1 = torch.Generator().manual_seed(9100 + seed)
    g2 = torch.Generator().manual_seed(9200 + seed)

    copier = (
        real + 0.02 * torch.randn(real.shape, generator=g1)
    ).clamp(0, 1)

    mean_noise = (
        Xtr.mean(dim=0, keepdim=True)
        + 0.1 * torch.randn(real.shape, generator=g2)
    ).clamp(0, 1)

    z = draw_prior(seed, N_SAMPLES)

    with torch.no_grad():
        collapsed = vae_samples(
            beta_models[(collapsed_config[seed], seed)], z
        )

        regular_vae = vae_samples(vae_models[seed], z)
        ae_prior = ae_models[seed].decode(z)

    image_sets = {
        "Real training": real,
        "Copier": copier,
        "Mean image + noise": mean_noise,
        "Collapsed VAE": collapsed,
        "VAE beta=1": regular_vae,
        "AE prior": ae_prior
    }

    for name, images in image_sets.items():
        scores = S.score_samples(images, fte, net, Xtr)

        metric_results.append({
            "seed": seed,
            "image_set": name,
            **scores
        })


metric_df = pd.DataFrame(metric_results)
display(metric_df.round(4))

,seed,image_set,frechet,precision,recall,nearest_train
0,0,Real training,2.1426,0.9331,0.8886,0.0000
1,0,Copier,2.2121,0.9331,0.8663,0.1336
2,0,Mean image + noise,138.4172,0.9387,0.0000,1.6170
3,0,Collapsed VAE,218.6080,1.0000,0.0000,1.5096
4,0,VAE beta=1,12.9154,0.9136,0.4150,1.1303
5,0,AE prior,70.8812,0.3510,0.8663,2.0602
6,1,Real training,2.0285,0.9415,0.9304,0.0000
7,1,Copier,2.1715,0.9443,0.9304,0.1333
8,1,Mean image + noise,138.8226,0.9610,0.0000,1.6145
9,1,Collapsed VAE,220.0111,1.0000,0.0000,1.5067


In [37]:
sizes = [30, 60, 120, 359, 1000]
train_features = S.get_features(net, Xtr)

results = []

for seed in SEEDS:
    rng = np.random.default_rng(11000 + seed)

    for size in sizes:
        for trial in range(10):
            idx = rng.choice(len(Xtr), size=size, replace=False)
            fd = S.frechet_distance(fte, train_features[idx])

            results.append({
                "seed": seed,
                "size": size,
                "frechet": fd
            })

df = pd.DataFrame(results)

summary = df.groupby(["seed", "size"])["frechet"].agg(["mean", "std"])

display(summary.round(4))

mean      std
seed size                  
0    30    24.9320   7.1726
     60    13.0120   6.2550
     120    5.1704   1.2143
     359    2.4721   0.5379
     1000   1.6891   0.2072
1    30    29.0279  14.6324
     60    12.4373   5.2277
     120    6.2799   1.6408
     359    3.1214   0.8132
     1000   1.6621   0.1978
2    30    24.4843   9.6761
     60    10.4250   3.3086
     120    4.9850   1.3364
     359    2.6531   0.8180
     1000   1.5965   0.0953

Fréchet distance measures how similar generated images are to real images, but it can miss copying and is affected by sample size. Precision measures how realistic generated images are, but it can give high scores to collapsed models that lack diversity. Recall measures diversity, but it does not guarantee that individual images look realistic. Nearest-training-image distance helps detect copying, but it cannot measure realism or diversity. Accuracy cannot be used because unconditional generated images have no ground-truth labels, and correctly recognizing a digit does not mean the image is realistic or diverse.